# N-Point Correlation Functions with SUGC

Three-point and N-point counting using the SUGC library. `sugc_weights` returns the alpha/beta correction weights for subvolume sub-sampling; `compute_3pcf_counts_with_sugc` and `compute_npoint_counts` call SUGC directly.

In [8]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np

import galform_analysis
from galform_analysis import set_base_dir, get_base_dir, SimulationConfig, load_redshift_mapping

mpl.setconfig()

# Configure the path to your GALFORM output directory once per session:
# set_base_dir('/cosma5/data/durham/<user>/Galform_Out/L800/model')
BASE_DIR = str(get_base_dir())
SIM = 'L800'

## 3PCF counts with SUGC

In [ ]:
from galform_analysis.analysis.correlation import (
    compute_3pcf_counts_with_sugc,
    compute_triplet_counts,
    sugc_weights,
    compute_npoint_counts,
    sugc_weights_npcf,
)
from galform_analysis.utils.read_galaxies import read_galaxy_positions
from galform_analysis.config import SimulationConfig
import os

IZ_NUM = 155
IVOL   = 0
iz_path = os.path.join(BASE_DIR, f'iz{IZ_NUM}')

sim = SimulationConfig(SIM)
pos, z = read_galaxy_positions(iz_path, ivol=IVOL, centrals_only=True)
print(f"Positions: {pos.shape},  z={z:.4f}")

rbins = np.logspace(-1, 1.0, 8)   # coarse bins for a quick example

# SUGC 3PCF: returns a dictionary containing decomposed triplet counts
T = compute_3pcf_counts_with_sugc(
    positions=pos,
    labels=np.zeros(len(pos), dtype=np.int64),   # single subvolume → all label 0
    rbins=rbins,
    m_selected=1,
    k_total=sim.n_subvolumes,
    boxsize=sim.box_size,
)
print(f"t_sss shape: {T['t_sss'].shape}  (r bins)")

Positions: (146786, 3),  z=1.4955
t_sss shape: (7,)  (r bins)


## Subvolume weights for the SCOPE/SUGC correction

In [6]:
# alpha-beta weights for m subvolumes drawn from k available
m = 4   # subvolumes used
k = sim.n_subvolumes  # total available

w_sss, w_ssd, w_ddd = sugc_weights(m, k)
print(f"w_sss = {w_sss:.6f}   (SSS configuration weight)")
print(f"w_ssd = {w_ssd:.6f}   (SSD configuration weight)")
print(f"w_ddd = {w_ddd:.6f}   (DDD configuration weight)")

# For N-point (N > 3)
for N in [3, 4]:
    weights = sugc_weights_npcf(N, m, k)
    print(f"N={N}: {len(weights)} weight terms")

w_sss = 0.000015   (SSS configuration weight)
w_ssd = 0.005203   (SSD configuration weight)
w_ddd = 2.658859   (DDD configuration weight)
N=3: 3 weight terms
N=4: 4 weight terms


## N-point counts (brute-force, small samples)

In [7]:
# Use a small subsample so the O(N^N) brute-force finishes quickly
rng   = np.random.default_rng(42)
idx   = rng.choice(len(pos), size=min(200, len(pos)), replace=False)
pos_s = pos[idx]

_, N4_counts = compute_npoint_counts(pos_s, rbins=rbins, N=4, boxsize=sim.box_size)
print(f"4-point counts shape: {N4_counts.shape}")

  i=0/200
4-point counts shape: (7,)
